# Construcción de un Clasificador de Texto Simple en PyTorch

Habiendo explorado los conceptos fundamentales para convertir el lenguaje humano en tensores aptos para máquinas, ahora estás listo para aplicar ese conocimiento a un desafío práctico. Has visto cómo el texto crudo se tokeniza, se transforma en IDs numéricos y luego se representa mediante vectores significativos llamados *embeddings* que capturan las relaciones semánticas entre palabras. Este laboratorio te guiará a través del proceso de construcción de un pipeline completo de clasificación de texto desde cero usando PyTorch. El objetivo es entrenar un modelo que pueda leer el título de una receta y clasificarla como basada en frutas o basada en vegetales.

Este laboratorio práctico consolidará tu comprensión de todo el flujo de trabajo, desde la preparación inicial de los datos hasta la evaluación final del modelo. Pondrás la teoría en práctica y abordarás desafíos comunes en el procesamiento de lenguaje natural (PLN).

Específicamente, en este laboratorio aprenderás a:

* **Preparar y preprocesar datos de texto crudo**, lo que incluye la limpieza, tokenización y construcción de un vocabulario a partir de un conjunto de datos de entrenamiento para evitar la fuga de datos (*data leakage*).
* **Construir objetos personalizados de PyTorch como `Dataset` y `DataLoader`**, e implementar funciones `collate` personalizadas para manejar eficientemente lotes de secuencias de texto de longitud variable.
* **Construir, entrenar y comparar múltiples arquitecturas de modelos**. Esto incluye un modelo eficiente que utiliza `nn.EmbeddingBag` y otros que realizan operaciones de agrupamiento (*pooling*) manuales como la media, el máximo y la suma.
* **Abordar un problema de desbalance de clases** en el conjunto de datos aplicando pesos de clase a la función de pérdida, una técnica vital para entrenar modelos justos y precisos con datos sesgados.
* **Evaluar tus modelos entrenados** para seleccionar el de mejor rendimiento basado en el puntaje F1 (F1 score) y luego probar su poder predictivo en títulos de recetas nuevos y nunca antes vistos.

## Imports

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import re
import pandas as pd

import helper_utils

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## El Conjunto de Datos de Recetas (Dataset)

Para este laboratorio, trabajarás con un conjunto de datos preparado especialmente, derivado de una gran colección de recetas. Cargarás el dataset preparado y lo alistarás para el proceso de modelado. Esto implica transformar los títulos de las recetas y las etiquetas en un formato numérico que un modelo de PyTorch pueda entender.

**La Colección de Recetas de Food.com**

Tus datos provienen del conjunto de datos [Food.com Recipes and User Interactions](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions), una vasta colección de más de 230,000 recetas recopiladas a lo largo de 18 años. Este rico dataset contiene desde nombres de recetas e ingredientes hasta pasos de cocina e información nutricional. Para los fines de este laboratorio, se ha pre-creado un subconjunto de estos datos, centrándose únicamente en recetas que son a base de frutas o a base de vegetales.

**Cómo se Creó el Subconjunto**

El archivo `recipes_fruit_veg.csv` que utilizarás fue generado por un script que filtró el conjunto de datos original. En resumen, el script realizó las siguientes acciones:

* Escaneó los ingredientes de cada receta buscando una lista predefinida de palabras clave comunes de frutas y vegetales.
* Para crear categorías distintas, solo conservó las recetas que contenían palabras clave de frutas pero ninguna de vegetales, o viceversa.
* Se excluyó cualquier receta que contuviera una mezcla de ambas categorías, o ninguna.

Este proceso garantiza que el conjunto de datos tenga dos clases mutuamente excluyentes, lo cual es ideal para esta tarea de clasificación. Si te interesa, puedes explorar la lógica exacta en la función `filter_recipe_dataset` dentro del archivo `helper_utils.py`.

* Ahora, ejecuta la celda de abajo para cargar el conjunto de datos. Esto leerá los datos del archivo `recipes_fruit_veg.csv` en un DataFrame de pandas llamado `df`.

In [ ]:
# Load the filtered dataset into a pandas DataFrame
df = pd.read_csv("recipes_fruit_veg.csv")

### Preparación de Entradas y Etiquetas

Comienza inspeccionando las primeras diez filas de tu DataFrame para comprender su estructura. Esto te ayudará a ver los datos de texto crudo con los que estarás trabajando.

In [ ]:
# Display the first 10 rows of the DataFrame
df.head(10)

De la tabla anterior, puedes ver las diferentes columnas que tienes disponibles. Para tu tarea de clasificación, **tu objetivo es predecir la categoría basándote en el nombre de la receta**.

* **Entrada del Modelo**: La columna `name` servirá como tu característica de entrada. Este es el texto que entrenarás al modelo para que lo comprenda.

* **Etiquetas del Modelo**: La columna `category` contiene las etiquetas ('fruit' o 'vegetable'). Sin embargo, los modelos de aprendizaje automático requieren datos numéricos, no texto. Tu siguiente paso es convertir estas etiquetas de cadena en números (por ejemplo, 0 y 1) para que el modelo pueda procesarlas.

* Realiza esta conversión creando una nueva columna `label` y estableciendo el valor predeterminado de **1 (para 'vegetable')** y luego actualizando la etiqueta a **0 para todas las recetas de 'fruit'**.

In [ ]:
# Crear la nueva columna 'label' y establecer un valor predeterminado.
# Establecer todo a 1 (la etiqueta para 'vegetable').
df['label'] = 1

# Usar indexación booleana para encontrar todas las filas donde la 'category' es 'fruit'
# Actualizar la 'label' en esas filas específicas a 0.
df.loc[df['category'] == 'fruit', 'label'] = 0

# Mostrar las primeras filas para confirmar que la nueva columna es correcta
df.head()

Con la columna numérica `label` lista, ya puedes extraer los datos a su formato final.

* Crea dos listas de Python separadas:
    * `texts`: Una lista que contenga los nombres de las recetas de la columna `name`.
    * `labels`: Una lista que contenga las etiquetas numéricas (0 o 1) de la columna `label`.

In [ ]:
# Conservar solo las filas que tengan un nombre de receta.
df_clean = df.dropna(subset=['name'])

# Obtener los nombres de las recetas como una lista.
texts = df_clean['name'].tolist()

# Obtener las etiquetas correspondientes como una lista.
labels = df_clean['label'].tolist()

* Ejecuta la siguiente celda para ver un desglose de las muestras con las que trabajarás.
    * **Nota**: Presta mucha atención al resultado. Notarás que hay significativamente menos recetas de frutas que de vegetales. Este es un problema común conocido como **desbalance de datos** (data imbalance), y es importante tenerlo en cuenta, ya que lo abordaremos más adelante en el cuaderno.

In [ ]:
# Verificar el tamaño final del dataset y la distribución de clases.
print(f"Total de muestras para clasificación:\t{len(texts)}")
print(f"Recetas de frutas:\t\t\t\t{labels.count(0)}, {round(labels.count(0)/(labels.count(0) + labels.count(1)) *100,1)} %")
print(f"Recetas de vegetales:\t\t\t{labels.count(1)}, {round(labels.count(1)/(labels.count(0) + labels.count(1)) *100,1)} %")

#### Previsualización de la Entrada y las Etiquetas

Tus datos ahora están estructurados con la columna `name` como la entrada del modelo y la columna `label` como su salida.

* Ejecuta la celda de abajo para revisar una muestra aleatoria de estos pares de entrenamiento.

In [ ]:
# Establecer el número de muestras aleatorias a mostrar.
num_samples = 10

# Mostrar una muestra de pares de nombres y etiquetas.
display(df[['name', 'label']].sample(num_samples, random_state=25).style.hide(axis="index"))

## División de los datos crudos

Para juzgar con precisión el rendimiento de tu modelo, debes probarlo con datos que **nunca haya visto antes**. Esto significa que tu primer paso, antes de cualquier procesamiento de texto o construcción de vocabulario, es separar tus listas de textos y etiquetas en dos grupos distintos.

La razón por la que haces esto ahora es para evitar la **fuga de datos** (*data leakage*). Si construyeras tu vocabulario a partir de **todo** el conjunto de datos, el modelo ya habría estado expuesto a las palabras de tus datos de validación. Esto no sería una prueba justa de cómo se desempeña ante información nueva del mundo real.

Al dividir los datos crudos primero, puedes construir el vocabulario utilizando únicamente las palabras de un grupo (los datos de entrenamiento). El otro grupo permanece completamente intacto, asegurando que sea un conjunto de datos verdaderamente "no visto" para tu evaluación final. Realizas esta división como un paso preliminar limpio antes de cualquier formato específico de PyTorch.

* Utiliza la función <code>[train_test_split()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)</code> de **scikit-learn** para gestionar esto.
    * `texts`: La lista de títulos de recetas que quieres dividir.
    * `labels`: La lista de etiquetas correspondientes para cada título.
    * `test_size=0.2`: Especifica que el 20% de los datos debe usarse para el conjunto de validación.
    * `stratify=labels`: Asegura que los conjuntos de entrenamiento y validación tengan las mismas proporciones de clases que el dataset original.

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    stratify=labels
)

# Imprimir el número de muestras en cada conjunto para verificar la división.
print(f"Muestras de entrenamiento: {len(train_texts)}")
print(f"Muestras de validación: {len(val_texts)}")

### Preprocesamiento de texto y construcción del vocabulario

Con tus datos ya divididos en grupos separados, es hora de convertir los títulos de las recetas a un formato numérico que tu modelo pueda entender. Esto implica limpiar el texto y luego construir un vocabulario para mapear cada palabra única a un número.

* Define la función `preprocess_text` y aplícala a los títulos de tus recetas.
    * Esta toma una cadena de texto crudo como entrada, la limpia convirtiéndola a minúsculas y elimina cualquier carácter que no sea letra o espacio.
    * Luego, tokeniza la cadena dividiéndola en una lista de palabras individuales.

In [ ]:
def preprocess_text(text):
    """Limpia y tokeniza una cadena de texto crudo.

    Args:
        text (str): El texto crudo a ser procesado.

    Returns:
        list: Una lista de palabras limpias (tokens).
    """
    # Convierte todo el texto a minúsculas.
    text = text.lower()
    # Elimina todos los caracteres que no sean letras o espacios en blanco.
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Divide la cadena limpia en una lista de palabras.
    words = text.split()

    return words

In [ ]:
# Preprocesar los textos de entrenamiento y de validación por separado.
processed_train_texts = [preprocess_text(text) for text in train_texts]
processed_val_texts = [preprocess_text(text) for text in val_texts]

* Define la clase `Vocabulary`, la cual crea y gestiona el mapeo entre palabras e IDs numéricos únicos.
    * `__init__`: Inicializa el vocabulario con tokens especiales para el relleno (`<pad>`) y palabras desconocidas (`<unk>`). También establece una frecuencia mínima (`min_freq`) para que una palabra sea incluida.
    * `build_vocab`: Escanea todos los textos procesados, cuenta la frecuencia de cada palabra y añade al mapeo del vocabulario las palabras que cumplen con el umbral de `min_freq`.
    * `encode`: Toma una lista de palabras y la convierte en la lista correspondiente de IDs numéricos. Utiliza el ID del token `<unk>` para cualquier palabra que no reconozca.
    * `__len__`: Permite encontrar el número total de elementos únicos en el vocabulario al llamar a `len()` sobre el objeto.

In [ ]:
class Vocabulary:
    """Construye y gestiona un vocabulario de palabra-a-índice a partir de texto.

    Atributos:
        word2idx (dict): Mapea palabras a enteros únicos.
        idx2word (dict): Mapea enteros de vuelta a palabras.
        min_freq (int): Frecuencia mínima de palabra para su inclusión.
    """
    def __init__(self, min_freq=1):
        """Inicializa el vocabulario.

        Args:
            min_freq (int): Frecuencia mínima de palabra para ser incluida.
        """
        # Mapeos para palabra-a-índice e índice-a-palabra con tokens especiales.
        self.word2idx = {'<pad>': 0, '<unk>': 1}
        self.idx2word = {0: '<pad>', 1: '<unk>'}
        self.min_freq = min_freq

    def build_vocab(self, texts):
        """Construye el vocabulario a partir de un corpus de textos tokenizados.

        Args:
            texts (lista de lista de str): Un corpus de oraciones tokenizadas.
        """
        # Cuenta la frecuencia de todas las palabras en el corpus.
        word_counts = Counter(word for text in texts for word in text)
        
        # Añade palabras al vocabulario si cumplen con la frecuencia mínima.
        for word, count in word_counts.items():
            if count >= self.min_freq:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, text):
        """Convierte un texto tokenizado en una secuencia de índices.

        Args:
            text (lista de str): Texto tokenizado a codificar.

        Returns:
            lista de int: Secuencia de índices correspondientes.
        """
        # Utiliza el token <unk> para palabras que no están en el vocabulario.
        return [self.word2idx.get(word, self.word2idx['<unk>']) for word in text]

    def __len__(self):
        """Devuelve el tamaño del vocabulario."""
        return len(self.word2idx)

* Crea un nuevo objeto de vocabulario para gestionar tus mapeos de palabra a número.
    * `min_freq=2`: Asegura que solo las palabras que aparezcan al menos dos veces en tus textos se añadan al vocabulario.

In [ ]:
vocab = Vocabulary(min_freq=2)

* Ejecuta el proceso principal de construcción del vocabulario. **El vocabulario se construirá utilizando únicamente los datos de entrenamiento**.
    * Escanea todos los `processed_texts`, cuenta la frecuencia de cada palabra y crea el mapeo final de palabra a número para todas las palabras que cumplan con la regla de `min_freq`.

In [ ]:
# Construir el vocabulario utilizando ÚNICAMENTE los textos de entrenamiento procesados.
vocab.build_vocab(processed_train_texts)

In [ ]:
print("Número de palabras en el vocabulario:", len(vocab))

* Utiliza el método `vocab.encode()` para convertir tus textos de entrenamiento preprocesados en listas de IDs numéricos.
* Repite exactamente el mismo proceso para tus textos de validación, utilizando el mismo vocabulario que construiste a partir de los datos de entrenamiento.

In [ ]:
# Codificar tanto los textos de entrenamiento como los de validación utilizando el vocabulario.
indexed_train_texts = [vocab.encode(text) for text in processed_train_texts]
indexed_val_texts = [vocab.encode(text) for text in processed_val_texts]

### Preparación de los Datos para el Entrenamiento

Ahora que tienes listas codificadas y separadas para tus datos de entrenamiento y validación, puedes crear objetos `Dataset` personalizados de PyTorch para cada una, con el fin de encapsular tus "textos indexados" y "etiquetas".

#### Construcción de tu `TextDataset` Personalizado

* Define un dataset personalizado heredando de la clase `Dataset` de PyTorch.
    * `__init__`: Almacena las listas de textos indexados y etiquetas numéricas. También crea un atributo `.classes` encontrando y ordenando los valores únicos de la lista de etiquetas.
    * `__len__`: Devuelve el número total de muestras (la longitud de tu lista de textos) en el dataset.
    * `__getitem__`: Recupera una única muestra de datos. Dado un índice `idx`, obtiene el texto y la etiqueta correspondientes, los convierte en tensores de PyTorch y los devuelve en un diccionario.

In [ ]:
class TextDataset(Dataset):
    """
    Un Dataset de PyTorch personalizado para manejar datos de texto y etiquetas.

    Esta clase encapsula un conjunto de datos de textos y sus etiquetas correspondientes,
    haciéndolo compatible con el DataLoader de PyTorch.
    """
    def __init__(self, texts, labels):
        """
        Inicializa el objeto TextDataset.

        Args:
            texts: Una lista o array de secuencias de texto numerizadas.
            labels: Una lista o array de etiquetas correspondientes.
        """
        # Almacena la colección de textos.
        self.texts = texts
        # Almacena la colección de etiquetas.
        self.labels = labels
        # Encuentra las etiquetas de clase únicas y las almacena.
        self.classes = sorted(list(set(labels)))

    def __len__(self):
        """
        Devuelve el número total de muestras en el conjunto de datos.
        """
        # Devuelve el tamaño del dataset basado en el número de textos.
        return len(self.texts)

    def __getitem__(self, idx):
        """
        Recupera una sola muestra del conjunto de datos en un índice dado.

        Args:
            idx: El índice de la muestra a recuperar.

        Returns:
            Un diccionario que contiene el texto y la etiqueta como tensores de PyTorch.
        """
        # Crea un diccionario para la muestra en el índice especificado.
        sample = {
            'text': torch.tensor(self.texts[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }
        
        # Devuelve el diccionario de la muestra.
        return sample

* Crea instancias de tu `TextDataset` para los conjuntos de entrenamiento y validación.

In [ ]:
# Crear los conjuntos de datos de entrenamiento y validación directamente a partir de los datos divididos.
train_dataset = TextDataset(indexed_train_texts, train_labels)
val_dataset = TextDataset(indexed_val_texts, val_labels)

# Imprimir el número de muestras en cada conjunto para verificar.
print(f"Muestras de entrenamiento:   {len(train_dataset)}")
print(f"Muestras de validación:      {len(val_dataset)}")  

#### La función Collate y los DataLoaders

En todos tus módulos anteriores sobre visión artificial, viste que cada imagen se preprocesaba a un **tamaño fijo y uniforme** (por ejemplo, 224x224 píxeles). Dado que cada tensor de imagen tenía exactamente las mismas dimensiones, el `DataLoader` podía combinarlos fácilmente en un solo lote.

El texto, sin embargo, es naturalmente variable. Por ejemplo:

* `"apple pie"` se convierte en un tensor de longitud 2.

* `"roasted broccoli and garlic"` se convierte en un tensor de longitud 4.



Aquí es donde te encontrarás con un problema con la **configuración predeterminada** del `DataLoader`. Por defecto, intenta crear un lote apilando los tensores individuales. Esta operación fallará porque tus tensores de texto tienen diferentes longitudes y no se pueden apilar en un único tensor uniforme. Este es el nuevo desafío que debes resolver.

**La solución: El parámetro `collate_fn`**

Para solucionar esto, utilizarás un parámetro opcional pero potente de la clase `DataLoader`: `collate_fn`.

Este parámetro te permite proporcionar tu propia función personalizada que define exactamente cómo tomar una lista de muestras de tu `Dataset` y combinarlas en un solo lote. En lugar de usar el comportamiento por defecto, el `DataLoader` ejecutará tu lógica personalizada.

```python
DataLoader(dataset=...,
           batch_size=...,
           shuffle=...,
           collate_fn=... # <-- Tu función personalizada va aquí
          )
```

Básicamente, le estás diciendo al `DataLoader`: "No uses tu método de apilamiento por defecto; usa mis instrucciones específicas para formar un lote".



Esta práctica de usar una `collate_fn` es también el método estándar y más eficiente. Aunque podrías rellenar todas las oraciones desde el principio, ese enfoque es ineficiente y conlleva un **desperdicio significativo de memoria y computación**. Al usar una `collate_fn` para realizar un "relleno dinámico", cada lote solo se rellena hasta la longitud de la oración más larga dentro de ese lote, lo que ahorra recursos y acelera el entrenamiento.

Definirás dos funciones de este tipo, una para cada una de las arquitecturas de modelo que se cubrirán en la siguiente sección.

* Primero, define `collate_batch_embeddingbag`, que toma una lista de muestras y prepara un lote formateado específicamente para la eficiente capa `nn.EmbeddingBag` de PyTorch (esto se cubrirá en la siguiente sección).
    * Primero extrae todas las etiquetas y los tensores de texto del lote.
    * Luego crea un único tensor 1D largo, `flattened_text`, concatenando todos los tensores de texto individuales.
    * Calcula un tensor de desplazamientos (`offsets`). Este tensor marca el índice de inicio de cada título de receta original dentro del tensor `flattened_text`.
    * Finalmente, devuelve el texto aplanado, los desplazamientos y las etiquetas, todos movidos al dispositivo activo (por ejemplo, tu GPU).

In [ ]:
def collate_batch_embeddingbag(batch):
    """
    Formatea un lote para nn.EmbeddingBag aplanando los textos y creando desplazamientos (offsets).

    Args:
        batch (lista de dict): Una lista de muestras del Dataset.

    Returns:
        tuple: Una tupla de tensores (flattened_text, offsets, labels).
    """
    # Extraer etiquetas de cada elemento y crear un único tensor.
    labels = torch.tensor([item['label'] for item in batch])
    # Extraer los tensores de texto individuales del lote.
    texts = [item['text'] for item in batch]
    # Crear una lista de las longitudes de cada texto, anteponiendo un 0.
    offsets = [0] + [len(text) for text in texts]
    # Convertir a un tensor de desplazamientos que represente el índice de inicio de cada secuencia.
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    # Concatenar todos los tensores de texto en un único tensor 1D largo.
    flattened_text = torch.cat(texts)
    
    # Devolver los tres tensores requeridos para el modelo EmbeddingBag.
    return flattened_text.to(device), offsets.to(device), labels.to(device)

* A continuación, define `collate_batch_manual`, que prepara un lote utilizando la técnica más común de **relleno** (*padding*).
    * Comienza extrayendo todas las etiquetas y los tensores de texto del lote.
    * Luego encuentra la longitud de la secuencia más larga (`max_len`) **dentro del lote actual**.
    * Se crea un nuevo tensor de ceros, `padded_texts`, con la forma `(batch_size, max_len)`. Esto actúa como una plantilla para el lote.
    * Después, recorre y copia cada secuencia de texto en el tensor `padded_texts`, dejando el espacio restante como relleno de ceros para las secuencias más cortas.
    * Finalmente, devuelve el tensor rectangular único `padded_texts` y el tensor `labels`, ambos movidos al dispositivo activo.

In [ ]:
def collate_batch_manual(batch):
    """
    Formatea un lote rellenando (padding) los textos hasta la misma longitud.

    Args:
        batch (lista de dict): Una lista de muestras del Dataset.

    Returns:
        tuple: Una tupla de tensores (padded_texts, labels).
    """
    # Extraer etiquetas de cada elemento y crear un único tensor.
    labels = torch.tensor([item['label'] for item in batch])
    # Extraer los tensores de texto individuales del lote.
    texts = [item['text'] for item in batch]
    # Encontrar la longitud de la secuencia más larga en este lote específico.
    max_len = max(len(text) for text in texts)
    # Crear un tensor de ceros para contener el lote con relleno.
    padded_texts = torch.zeros(len(texts), max_len, dtype=torch.long)
    # Copiar cada secuencia de texto en la fila correspondiente del tensor con relleno.
    for i, text in enumerate(texts):
        padded_texts[i, :len(text)] = text
        
    # Devolver los textos con relleno y las etiquetas, movidos al dispositivo activo.
    return padded_texts.to(device), labels.to(device)

* Crea dos instancias de `DataLoader`, `train_loader_embag` y `val_loader_embag`.
    * `collate_fn=collate_batch_embeddingbag`: Pasando tu función personalizada `collate_batch_embeddingbag` a ambos Dataloaders.

In [ ]:
# Establecer el número de muestras a procesar en cada lote.
batch_size = 32

# Crear el DataLoader para el conjunto de entrenamiento con `collate_batch_embeddingbag`
train_loader_embag = DataLoader(train_dataset, 
                                batch_size=batch_size, 
                                shuffle=True, 
                                collate_fn=collate_batch_embeddingbag
                               )

# Crear el DataLoader para el conjunto de validación con `collate_batch_embeddingbag`
val_loader_embag = DataLoader(val_dataset, 
                              batch_size=batch_size, 
                              shuffle=False, 
                              collate_fn=collate_batch_embeddingbag
                             )

* Ahora, crea dos instancias de `DataLoader`, `train_loader_manual` y `val_loader_manual`.
    * `collate_fn=collate_batch_manual`: Pasando tu función personalizada `collate_batch_manual` a ambos Dataloaders.

In [ ]:
# Crear el DataLoader para el conjunto de entrenamiento con `collate_batch_manual`
train_loader_manual = DataLoader(train_dataset, 
                                 batch_size=batch_size, 
                                 shuffle=True, 
                                 collate_fn=collate_batch_manual
                                )

# Crear el DataLoader para el conjunto de validación con `collate_batch_manual`
val_loader_manual = DataLoader(val_dataset, 
                               batch_size=batch_size, 
                               shuffle=False, 
                               collate_fn=collate_batch_manual
                              )

## Arquitecturas de Modelos: EmbeddingBag vs. Pooling Manual

Hasta ahora, has completado todo el pipeline de datos: has cargado, limpiado y tokenizado el texto; has construido un vocabulario numérico; y has preparado los `DataLoaders` para alimentar los lotes a un modelo. Ahora, es el momento de diseñar los modelos que aprenderán de estos datos.

Vas a construir y comparar cuatro arquitecturas de modelos diferentes. Los modelos comienzan convirtiendo los índices de las palabras en vectores de incrustación (embeddings) densos, pero difieren en el siguiente paso: cómo agregan los embeddings de una oración de longitud variable en un único vector de tamaño fijo que pueda usarse para la clasificación.

1. El primer modelo utiliza `nn.EmbeddingBag`, una capa integrada de PyTorch altamente eficiente que realiza una agregación por **media** (*mean*) en un solo paso.

2. Los siguientes modelos utilizan un enfoque de **pooling manual**. Utilizaremos tres enfoques donde agregaremos los embeddings mediante la **media** (*mean*), el **máximo** (*max*) y la **suma** (*sum*).

Al construir y probar estos modelos, descubrirás cuál de las nuevas estrategias funciona mejor para esta tarea específica.

### EmbeddingBagClassifier

Tu primer modelo, el `EmbeddingBagClassifier`, es una arquitectura simple pero potente construida alrededor de la eficiente capa `nn.EmbeddingBag` de PyTorch.

* `__init__`: Define las tres capas del modelo:
    * <code>[nn.EmbeddingBag](https://docs.pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html)</code>: Este es el núcleo del modelo. En un solo paso, esta capa toma tus IDs de tokens y sus desplazamientos (offsets) correspondientes y calcula un vector de tamaño fijo para cada título de receta promediando sus embeddings de palabras.
    * `nn.Dropout`: Esta es una capa de regularización estándar que ayuda a evitar que el modelo sufra de sobreajuste (*overfitting*).
    * `nn.Linear`: Esta es la capa de clasificación final totalmente conectada que toma el vector del embedding bag y genera las puntuaciones brutas para cada clase ('fruta' o 'verdura').
* El método `forward` define el flujo de datos real. Los datos pasan a través de las capas en este orden: `EmbeddingBag` → `Dropout` → `Linear`.

In [ ]:
class EmbeddingBagClassifier(nn.Module):
    """
    Un clasificador de texto simple que utiliza una capa nn.EmbeddingBag.

    Args:
        vocab_size (int): El tamaño del vocabulario.
        embedding_dim (int): El tamaño de los vectores de incrustación (embeddings).
        num_classes (int): El número de clases de salida.
    """
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()
        # La capa central que calcula eficientemente los embeddings para secuencias de longitud variable.
        # 'mode=mean' especifica que promediará los embeddings de todas las palabras en una secuencia.
        self.embedding_bag = nn.EmbeddingBag(vocab_size, embedding_dim, mode='mean')
        # Una capa de dropout estándar para regularización y así prevenir el sobreajuste.
        self.dropout = nn.Dropout(0.5)
        # La capa final totalmente conectada que mapea el embedding a las clases de salida.
        self.fc = nn.Linear(embedding_dim, num_classes)

    def forward(self, text, offsets=None):
        """
        Define el paso hacia adelante (forward pass) del modelo.

        Args:
            text (torch.Tensor): Un tensor 1D de índices de texto concatenados.
            offsets (torch.Tensor): Un tensor 1D de posiciones de inicio para cada secuencia.

        Returns:
            torch.Tensor: Las puntuaciones de salida brutas (logits) para cada clase.
        """
        # Calcular el vector de embedding único para cada secuencia en el lote.
        embedded = self.embedding_bag(text, offsets)
        # Aplicar dropout a los embeddings.
        embedded = self.dropout(embedded)
        # Pasar el resultado a través de la capa lineal final para obtener las puntuaciones de clase.
        return self.fc(embedded)

### ManualPoolingClassifier

Tus otros modelos, representados por `ManualPoolingClassifier`, adoptan un enfoque más directo donde definirás explícitamente la lógica para agregar las incrustaciones (embeddings) de las palabras.

* `__init__`: Define las capas y la configuración del modelo:
    * `nn.Embedding`: Una capa de embedding estándar que convierte los IDs de los tokens de tu texto con relleno (padded) en vectores densos.
        * Se establece `padding_idx=0` para asegurar que los tokens de relleno sean ignorados durante el entrenamiento.
    * `self.pooling`: Esto no es una capa, sino un atributo que almacena el nombre de la estrategia de pooling (`'mean'`, `'max'` o `'sum'`) que deseas utilizar.
    * `nn.Dropout`: Una capa de regularización estándar.
    * `nn.Linear`: La capa de clasificación final totalmente conectada.
* El método `forward` define un flujo de datos más complejo:
    * Primero, obtiene los embeddings y crea una **máscara** (mask) para asegurar que los tokens de relleno se pongan a cero y no afecten los cálculos de pooling.
    * El bloque `if/elif` utiliza entonces la estrategia de `pooling` elegida para agregar los embeddings de las palabras en un único vector de tamaño fijo para cada título de receta. Debido a esta lógica, **esta única clase actúa como un plano para tres variaciones de modelos distintos**.
    * Finalmente, el vector resultante del pooling se pasa a través de `Dropout` y la capa `Linear` final para obtener la predicción.

In [ ]:
class ManualPoolingClassifier(nn.Module):
    """
    Un clasificador de texto que utiliza una capa nn.Embedding seguida de una
    estrategia de pooling manual (mean, max o sum).

    Args:
        vocab_size (int): El tamaño del vocabulario.
        embedding_dim (int): El tamaño de los vectores de incrustación (embeddings).
        num_classes (int): El número de clases de salida.
        pooling (str, opcional): La estrategia de pooling a utilizar.
                                 Opciones: 'mean', 'max', 'sum'.
                                 Por defecto es 'mean'.
    """
    def __init__(self, vocab_size, embedding_dim, num_classes, pooling='mean'):
        super().__init__()
        # Capa de embedding que mapea IDs de tokens a vectores.
        # padding_idx=0 asegura que el token de relleno sea ignorado durante el entrenamiento.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # Almacenar la estrategia de pooling elegida.
        self.pooling = pooling
        # Capa final totalmente conectada para la clasificación.
        self.fc = nn.Linear(embedding_dim, num_classes)
        # Capa de dropout para regularización.
        self.dropout = nn.Dropout(0.5)

    def forward(self, text):
        """
        Define el paso hacia adelante (forward pass) del modelo.

        Args:
            text (torch.Tensor): Un lote de índices de texto con relleno (padded).

        Returns:
            torch.Tensor: Las puntuaciones de salida brutas (logits) para cada clase.
        """
        # Obtener los embeddings para el texto de entrada.
        # Forma: (batch_size, max_len, embedding_dim)
        embedded = self.embedding(text)

        # Crear una máscara para ignorar los tokens de relleno en los cálculos de pooling.
        # La máscara tendrá 1s para tokens reales y 0s para tokens de relleno.
        mask = (text != 0).float().unsqueeze(-1)
        # Aplicar la máscara mediante multiplicación elemento a elemento.
        embedded = embedded * mask

        # Aplicar la estrategia de pooling elegida.
        if self.pooling == 'mean':
            # Sumar embeddings y dividir por el número real de tokens no rellenados.
            # clamp(min=1) evita la división por cero para secuencias vacías.
            pooled = embedded.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        elif self.pooling == 'max':
            # Establecer las posiciones rellenadas a infinito negativo para que max() las ignore.
            embedded[mask.squeeze(-1) == 0] = float('-inf')
            pooled, _ = embedded.max(dim=1)
        elif self.pooling == 'sum':
            # Sumar los embeddings de todos los tokens que no son de relleno.
            pooled = embedded.sum(dim=1)

        # Aplicar dropout y la capa lineal final.
        pooled = self.dropout(pooled)
        return self.fc(pooled)

### Inicialización de los Modelos

* Define los hiperparámetros para los modelos.
    * `vocab_size`: El número total de palabras únicas en tu vocabulario, utilizado para establecer el tamaño de la capa de embedding.
    * `embedding_dim`: Un hiperparámetro que tú eliges y que define el tamaño del vector utilizado para representar cada palabra.
    * `num_classes`: El número de categorías de salida que tu modelo predecirá (en este caso, 2).

In [ ]:
# Define Model Hyperparameters
vocab_size = len(vocab)
embedding_dim = 64
num_classes = 2

* Inicializa el modelo EmbeddingBag.

In [ ]:
model_embag = EmbeddingBagClassifier(vocab_size, embedding_dim, num_classes)

* A continuación, inicializa las tres variantes del `ManualPoolingClassifier`, una para cada estrategia de pooling: `'mean'`, `'max'`, y `'sum'`.

In [ ]:
# Initialize the 'mean' pooling variant
model_manual_mean = ManualPoolingClassifier(vocab_size, embedding_dim, num_classes, pooling='mean')

# Initialize the 'max' pooling variant
model_manual_max = ManualPoolingClassifier(vocab_size, embedding_dim, num_classes, pooling='max')

# Initialize the 'sum' pooling variant
model_manual_sum = ManualPoolingClassifier(vocab_size, embedding_dim, num_classes, pooling='sum')

## Ejecución del Experimento de Entrenamiento

Con las arquitecturas de tus modelos y los cargadores de datos ya configurados, es hora de comenzar el proceso de entrenamiento. Sin embargo, antes de empezar, hay un problema crucial que abordar que se notó durante la exploración inicial de los datos.

Anteriormente, viste que el conjunto de datos está bastante **desequilibrado**, con significativamente más recetas de verduras que de frutas.

Si entrenas el modelo con este conjunto de datos tal cual, verá la clase 'verdura' con mucha más frecuencia. Esto puede causar que el modelo se vuelva sesgado, aprendiendo simplemente a predecir la clase mayoritaria ('verdura') la mayor parte del tiempo para lograr una alta precisión, mientras que tendrá un mal desempeño en la clase minoritaria 'fruta' que rara vez ve.

Para solucionar esto, utilizarás una estrategia común y efectiva: **pesos de clase** (*class weights*). La idea es dar más importancia a la clase subrepresentada durante el entrenamiento. Al asignar un peso más alto a la clase 'fruta', le indicas a la función de pérdida que penalice al modelo con más fuerza por cometer errores en las recetas de frutas. Esto obliga al modelo a prestar más atención a la clase minoritaria, lo que conduce a un clasificador más equilibrado y justo.

### Abordando el Desequilibrio de Clases

* El primer paso es obtener una lista de todas las etiquetas que pertenecen a tu **conjunto de entrenamiento** para calcular los pesos de las clases.
* Dado que ya utilizaste `train_test_split` anteriormente, esta lista ya está disponible en la variable `train_labels`.

In [ ]:
train_labels_list = train_labels

* Utiliza <code>[compute_class_weight()](https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html)</code>, una función auxiliar de la **biblioteca scikit-learn** para calcular automáticamente los pesos adecuados para contrarrestar tu conjunto de datos desequilibrado.
    * `class_weight='balanced'`: Esta es la instrucción clave. Le indica a la función que calcule automáticamente pesos que sean inversamente proporcionales a la frecuencia con la que aparece cada clase.
    * `classes=np.unique(train_labels_list)`: Este parámetro proporciona a la función una lista de todas las etiquetas de clase únicas que existen en tus datos, que son `[0, 1]`.
    * `y=train_labels_list`: Esta es la lista de todas las etiquetas de tu conjunto de entrenamiento. La función **descubre automáticamente** cuál es la clase minoritaria contando las ocurrencias de cada etiqueta en esta lista. Estos conteos se utilizan luego para calcular los pesos "balanceados" finales.
        * En un escenario con más de dos clases, este proceso funciona exactamente igual. La función calcularía un peso único para cada clase basado en su frecuencia. Si hubiera varias clases minoritarias, cada una recibiría un peso similarmente alto.

In [ ]:
# Utilizar la utilidad de scikit-learn para calcular automáticamente los pesos de las clases.
class_weights = compute_class_weight(
    # La estrategia para calcular los pesos. 'balanced' es automática.
    class_weight='balanced',
    # El array de etiquetas de clase únicas (por ejemplo, [0, 1]).
    classes=np.unique(train_labels_list),
    # La lista de todas las etiquetas de entrenamiento, utilizada para contar las frecuencias de las clases.
    y=train_labels_list
)

* Convierte los pesos calculados por scikit-learn al formato requerido por PyTorch.
* Los pesos se convierten primero de un array de NumPy a un **tensor** de PyTorch con un tipo de datos `float`.
* El método `.to(device)` mueve entonces este tensor a tu dispositivo de entrenamiento activo (por ejemplo, 'cuda').

In [ ]:
# Convertir el array de NumPy de pesos en un tensor de PyTorch de tipo float
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Imprimir los pesos finales para verificar el cálculo.
print("Pesos de Clase Calculados:")
print(f"  - Fruta (Clase 0):     {class_weights[0]:.2f}")
print(f"  - Verdura (Clase 1):   {class_weights[1]:.2f}")

### Configuración de la Función de Pérdida

* Define `nn.CrossEntropyLoss` como tu función de pérdida.
    * El paso importante aquí es que estás pasando tu tensor `class_weights` calculado previamente al parámetro `weight`.
        * Esto instruye a la función de pérdida a penalizar los errores en la clase minoritaria 'fruta' con más fuerza que los errores en la clase mayoritaria 'verdura', obligando al modelo a aprender de ambas clases de manera más equitativa.

In [ ]:
# Inicializar la función CrossEntropyLoss con los `class_weights` calculados.
loss_function = nn.CrossEntropyLoss(weight=class_weights)

### Entrenamiento de Todas las Variantes del Modelo

Ahora puedes comenzar a entrenar tus modelos. La función `training_loop` se encarga del proceso de entrenamiento y devuelve el modelo entrenado junto con un diccionario de las métricas finales de validación (`val_accuracy`, `val_precision`, `val_recall` y `val_f1`).

El bucle de entrenamiento es muy similar a lo que has visto antes, con una diferencia clave para manejar los dos tipos de modelos que has creado:

* Comprueba el nombre del modelo que se está entrenando para determinar cómo despaquetar los lotes de datos del `DataLoader`.
* Si es un `EmbeddingBagClassifier`, espera un lote que contenga `text`, `offsets` y `labels`, que es la salida de la función `collate_batch_embeddingbag`.
* Si es un `ManualPoolingClassifier`, espera un lote con solo el `text` con relleno (*padded*) y las `labels`, que es la salida de la función `collate_batch_manual`.

Esto permite que esta única función entrene de manera flexible las cuatro variaciones de tu modelo.

In [ ]:
# ### Descomentar si quieres ver la función del bucle de entrenamiento

# helper_utils.display_function(helper_utils.training_loop)

* Primero, establece el número de épocas de entrenamiento. Este valor se utilizará para todas las ejecuciones de entrenamiento de los modelos para asegurar que cada uno se entrene durante la misma duración.

In [ ]:
num_epochs = 5

* Ejecuta el bucle de entrenamiento para el `EmbeddingBagClassifier`.
    * Observa que estás pasando el `train_loader_embag` y el `val_loader_embag` a la función, los cuales fueron creados usando la `collate_fn` específica requerida por este modelo.

In [ ]:
# Realizar el entrenamiento para el EmbeddingBagClassifier.
trained_embag, results_embag = helper_utils.training_loop(
    model_embag, 
    train_loader_embag, 
    val_loader_embag, 
    loss_function,
    num_epochs, 
    device
)

# Mostrar los resultados 
print("\nModelo: EmbeddingBagClassifier")
helper_utils.print_final_metrics(results_embag)

* Entrena el `ManualPoolingClassifier` que utiliza la estrategia de pooling `'mean'`.
    * Este modelo, junto con las otras variantes manuales, utilizará el `train_loader_manual` y el `val_loader_manual`, que proporcionan lotes de datos con relleno (padded).

In [ ]:
# Realizar el entrenamiento para el ManualPoolingClassifier, variante `mean`.
trained_mean, results_mean = helper_utils.training_loop(
    model_manual_mean, 
    train_loader_manual, 
    val_loader_manual, 
    loss_function, 
    num_epochs, 
    device
)

# Mostrar los resultados 
print("\nModelo: ManualPoolingClassifier (MEAN)")
helper_utils.print_final_metrics(results_mean)

* A continuación, entrena el `ManualPoolingClassifier` que utiliza la estrategia de pooling `'max'`.

In [ ]:
# Realizar el entrenamiento para el ManualPoolingClassifier, variante `max`.
trained_max, results_max = helper_utils.training_loop(
    model_manual_max, 
    train_loader_manual, 
    val_loader_manual, 
    loss_function, 
    num_epochs, 
    device
)

# Mostrar los resultados 
print("\nModelo: ManualPoolingClassifier (MAX)")
helper_utils.print_final_metrics(results_max)

* Finalmente, entrena el `ManualPoolingClassifier` que utiliza la estrategia de pooling `'sum'`.

In [ ]:
# Realizar el entrenamiento para el ManualPoolingClassifier, variante `sum`.
trained_sum, results_sum = helper_utils.training_loop(
    model_manual_sum, 
    train_loader_manual, 
    val_loader_manual, 
    loss_function, 
    num_epochs, 
    device
)

# Mostrar los resultados 
print("\nModelo: ManualPoolingClassifier (SUM)")
helper_utils.print_final_metrics(results_sum)

Ahora puedes comparar el rendimiento de los cuatro modelos basándote en sus métricas de validación finales.

In [ ]:
results_df = helper_utils.get_results_df(
    results_embag,
    results_mean,
    results_max,
    results_sum
)

results_df

## Selección y Evaluación del Mejor Modelo

Hasta ahora, has completado la fase experimental entrenando cuatro arquitecturas de modelos diferentes. Cada una se entrenó bajo las mismas condiciones, utilizando pesos de clase para manejar el conjunto de datos desequilibrado. Esta "competencia" te ha proporcionado métricas de rendimiento para cada enfoque.

Ahora, es el momento de pasar de la experimentación general a la finalización y evaluación enfocada.

* Reúne todos tus modelos entrenados y sus resultados en un único diccionario llamado `all_trained_data`.

In [ ]:
all_trained_data = {
    'EmbeddingBag': (trained_embag, results_embag),
    'Manual (mean)': (trained_mean, results_mean),
    'Manual (max)': (trained_max, results_max),
    'Manual (sum)': (trained_sum, results_sum)
}

### Comparación del Rendimiento del Modelo

Dado que tienes significativamente más recetas de verduras que de frutas, la precisión (*accuracy*) por sí sola puede ser una métrica engañosa. Un modelo podría lograr una precisión alta simplemente adivinando la clase mayoritaria ('verdura') en cada ocasión. El **F1-score**, sin embargo, es una métrica más robusta para este caso de uso porque calcula un equilibrio entre la precisión (*precision*) y la exhaustividad (*recall*). Un F1-score alto indica que el modelo está funcionando bien tanto en la clase mayoritaria como en la minoritaria, que es exactamente lo que buscas.

* Utiliza la función `plot_and_select_best_model` para comparar los cuatro modelos entrenados basándote únicamente en su F1-score de validación.
    * Esta función visualizará los resultados en un gráfico de barras y luego devolverá la instancia del modelo que alcanzó el F1-score más alto.

In [ ]:
best_model = helper_utils.plot_and_select_best_model(all_trained_data)

### Probando el Mejor Modelo con Nuevos Ejemplos

Has hecho el trabajo pesado: entrenaste múltiples arquitecturas y seleccionaste sistemáticamente el mejor modelo.

Ahora viene la prueba final. Es hora de ver cómo se comporta tu mejor modelo con datos completamente nuevos y desconocidos. Esta es la mejor manera de obtener una sensación cualitativa de qué tan bien ha aprendido el modelo a generalizar.

* Define una lista `test_products` que contenga una mezcla de nuevos títulos de recetas. Esta lista incluye ejemplos directos, así como otros más desafiantes para ver dónde destaca el modelo y dónde podría tener dificultades.
    * ¡Siéntete libre de añadir tus propios títulos de recetas a esta lista para poner a prueba el modelo aún más!

**Nota**: Recuerda que las predicciones del modelo se basan *únicamente* en las palabras del nombre (`name`) de la receta. Nunca se le mostró la lista de ingredientes, por lo que no tiene conocimiento de si las frutas o las verduras son el ingrediente dominante. El nombre de una receta a veces puede ser engañoso, y la clasificación del modelo reflejará solo lo que ha aprendido del texto del título.

In [ ]:
test_products = [
    "Blueberry Muffins",                  # Expected: Fruit
    "Spinach and Feta Stuffed Chicken",   # Expected: Vegetable
    "Classic Carrot Cake with Frosting",  # Expected: Vegetable
    "Tomato and Basil Bruschetta",        # Expected: Vegetable
    "Avocado Toast",                      # Expected: Fruit
    "Zucchini Bread with Walnuts",        # Expected: Vegetable
    "Lemon and Herb Roasted Chicken",     # Expected: Fruit
    "Strawberry Rhubarb Pie",             # Expected: Fruit
]

* Finalmente, recorre la lista `test_products` para ejecutar la predicción de cada receta y ver el resultado final del modelo.

In [ ]:
## Descomentar si quieres ver la función de predicción de categoría

# helper_utils.display_function(helper_utils.predict_category)

In [ ]:
# Recorrer cada producto de prueba
for product in test_products:
    # Llamar a la función de predicción con los argumentos requeridos
    category = helper_utils.predict_category(
        best_model,
        product,
        vocab,
        preprocess_text,
        device
    )
    # Imprimir los resultados
    print(f"Producto: '{product}'\nPredicción: {category}.\n")

## Conclusión

¡Felicidades por completar este laboratorio! Has navegado con éxito por todo el pipeline para construir un clasificador de texto en PyTorch, pasando de texto bruto y no estructurado a un modelo predictivo funcional. Este laboratorio demostró cómo traducir los conceptos fundamentales de la representación de texto en una aplicación del mundo real.

Comenzaste preprocesando cuidadosamente los datos de las recetas, construyendo un vocabulario y preparando `DataLoaders` capaces de manejar la naturaleza variable de los datos de texto, una diferencia clave con respecto al trabajo con imágenes de tamaño fijo. Implementaste dos arquitecturas de modelo distintas pero relacionadas: una utilizando la capa altamente optimizada `nn.EmbeddingBag` y otra que permitía un control manual sobre diferentes estrategias de pooling (`mean`, `max` y `sum`). Esta comparación te brindó una visión directa de cómo los diferentes métodos de agregación pueden impactar en el rendimiento.

Además, abordaste el desafío común del desequilibrio de clases calculando y aplicando pesos de clase, asegurando que tu modelo aprendiera a clasificar eficazmente tanto las recetas de frutas como las de verduras, en lugar de sesgarse hacia la clase mayoritaria. Finalmente, evaluaste sistemáticamente tus modelos utilizando el F1-score para identificar al mejor y lo probaste con datos nuevos y desconocidos, que es la medida definitiva de la capacidad de generalización de un modelo.

El modelo que construiste sirve como una excelente línea base. Los embeddings se entrenaron desde cero en tu conjunto de datos específico. El siguiente paso lógico, que se basa directamente en estas habilidades, es aprovechar el poder de los **embeddings preentrenados**. El uso de vectores de modelos como GloVe o BERT, que han sido entrenados con miles de millones de palabras, puede proporcionar a tu modelo una comprensión del lenguaje mucho más rica desde el principio, lo que a menudo conduce a mejoras significativas en el rendimiento.